# Notebook 06: Cluster Profiling, Interpretation & Stability Validation
**Project**: Maternal-Child Health Profiles Across Countries Through Unsupervised Clustering

### Objectives:
1. Compute cluster-wise means, medians, and standardized z-score deviations.
2. Construct multi-dimensional radar charts across maternal, child, and WASH pillars.
3. Generate objective, statistically grounded epidemiological narratives.
4. Execute stability validation across 10 random seeds, bootstrap subsamples, and feature subsets.

In [ ]:
import sys
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

import config
from src.data_loader import get_latest_cross_sectional_dataset
from src.preprocessing import HealthDataPreprocessor
from src.interpretation import (
    compute_cluster_profiles,
    generate_cluster_narrative,
    get_cluster_membership,
    run_stability_analysis,
)

### 1. Compute Cluster Health Profiles (K=4)

In [ ]:
raw_data = get_latest_cross_sectional_dataset(min_year=2000)
preprocessor = HealthDataPreprocessor(max_missing_country_pct=35.0, imputation_strategy="median")
cleaned, imputed, standardized = preprocessor.fit_transform_pipeline(raw_data, save_artifacts=False)

X = standardized[config.FEATURE_COLUMNS].values
km = KMeans(n_clusters=4, random_state=config.RANDOM_SEED, n_init=config.N_INIT)
labels = km.fit_predict(X)

means_df, medians_df, z_df = compute_cluster_profiles(imputed, labels)
print("Cluster Means:")
print(means_df[["skilled_birth_attendance", "under_five_mortality_rate", "child_stunting_prevalence", "improved_sanitation_access"]])

### 2. Standardized Profile Heatmap

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(z_df.rename(columns=config.SHORT_TO_LABEL), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Cluster Standardized Deviations (Z-Scores) Across Health Indicators")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.tight_layout()
plt.show()

### 3. Country Membership & Epidemiological Narratives

In [ ]:
members_df = get_cluster_membership(standardized, labels)
for c in range(4):
    c_name = f"Cluster {c+1}"
    c_countries = members_df[members_df["Cluster"] == c]["CountryName"].tolist()
    narr = generate_cluster_narrative(c_name, z_df.loc[c_name], c_countries)
    print(f"\n=== {c_name} (N={narr['country_count']} countries) ===")
    print("  Maternal Care:", narr["maternal_health_profile"])
    print("  Child Mortality:", narr["child_mortality_profile"])
    print("  Child Nutrition:", narr["child_nutrition_profile"])
    print("  Countries:", narr["countries"])

### 4. Stability Validation Across Seeds and Subsets

In [ ]:
stability = run_stability_analysis(X, imputed, base_k=4)
print(f"Mean Seed ARI across 10 initializations: {stability['Mean_Seed_ARI']:.4f} (+/- {stability['Std_Seed_ARI']:.4f})")
print(f"Mean 85% Bootstrap Subsample ARI: {stability['Mean_Bootstrap_ARI']:.4f} (+/- {stability['Std_Bootstrap_ARI']:.4f})")
print("\nFeature Subsets Agreement:")
print(stability["Feature_Subsets_DF"])